### Draw Main Graph

In [1]:
import numpy as np
import tensorflow.compat.v1 as tf
from config import *
from GPT_Model import *
from data_pipeline import get_data, data_train_files, data_test_files

tf.reset_default_graph()

tf.compat.v1.disable_eager_execution()
X = tf.placeholder(tf.int32, [None, hparams.n_time])
Y = tf.placeholder(tf.int32, [None, hparams.n_time])

X_onehot = tf.one_hot(X, axis=2, depth=hparams.n_vocab[0])

logits = model(hparams, X)['logits']
probs = tf.nn.softmax(logits, axis=2)
cross_entropy = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=Y, logits=logits)
loss = tf.reduce_mean(cross_entropy)

temperature = 0
u = tf.random.uniform(shape=tf.shape(logits[:, -1]), minval=1e-5, maxval=1.-1e-5)
u = (logits[:, -1] - tf.log(temperature + 1e-8)) - tf.log(-tf.log(u))
sample = tf.argmax(u, axis=1)

'''
Train
'''
global_step = tf.Variable(0, name='global_step')
learning_rate = tf.Variable(1e-3, name='learning_rate')
train_step = tf.train.AdamOptimizer(learning_rate).minimize(loss, global_step)

'''
Session Open
'''

# GPU number to use
gpu_options = tf.GPUOptions(visible_device_list="0")
sess = tf.Session(config=tf.ConfigProto(gpu_options=gpu_options))

sess.run(tf.global_variables_initializer())

print('graph create')

Instructions for updating:
If using Keras pass *_constraint arguments to layers.
graph create


### Load model if exist

In [3]:
import tf_slim as slim
from tensorflow.python import pywrap_tensorflow

load_dir = '../save_model'
save_dir = '../save_model'

def get_variables_from_checkpoint_file(file_name):
    variables = []
    reader = pywrap_tensorflow.NewCheckpointReader(file_name)

    var_to_shape_map = reader.get_variable_to_shape_map()
    for key in sorted(var_to_shape_map):
        variables.append((key, var_to_shape_map[key]))

    return variables

saver = tf.train.Saver()

if True:
    restore_file = tf.train.latest_checkpoint(load_dir)
    print(restore_file)
    if restore_file is not None:
        try:
            saver.restore(sess, restore_file)
            print("Model restored.", restore_file)
        except:
            saved_variables = get_variables_from_checkpoint_file(restore_file)
            model_variables = slim.get_variables_to_restore()
            restore_variables = []
            for model_variable in model_variables:
                for saved_variable_name, saved_variable_shape in saved_variables:
                    model_variable_name = model_variable.name.split(":")[0]
                    if saved_variable_name == model_variable_name and tuple(saved_variable_shape) == model_variable.shape:
                        restore_variables.append(model_variable)

            init_saver = tf.train.Saver(restore_variables)
            init_saver.restore(sess, restore_file)
            print("Model partially restored.")
    else:
        print('model not exist.')
        

None
model not exist.


### TensorboardX Logger

In [4]:
from tensorboardX import SummaryWriter

class Logger(SummaryWriter):
    def __init__(self, logdir):
        super(Logger, self).__init__(logdir)

    def log(self, log_string, value, iteration):
            self.add_scalar(log_string, value, iteration)
            
logger = Logger(save_dir)            

### Train Loop

In [7]:
from IPython.display import clear_output
from tqdm import tqdm_notebook as tqdm
import matplotlib.pyplot as plt
from time import sleep
import time
import math

type = 'music'
# type = 'poetry'

print('iteration\t', 'loss\t', 'train_perplexity\t')
while(True):
    for _ in range(100):
        _inputs = []
        _targets = []
        for _ in range(batch_size):
            while(True):
                x, y = get_data(hparams.n_time, data_train_files,'train',0, type)
                if(x.shape == y.shape):
                    break
                 
            _inputs.append(x)
            _targets.append(y)
        _inputs = np.stack(_inputs)
        _targets = np.stack(_targets)
#         print(_inputs.shape, _targets.shape)
        
        _, _global_step, _loss = sess.run([train_step, global_step, loss], 
                                          feed_dict={X: _inputs, 
                                                     Y: _targets,
                                                     learning_rate: 1e-5})
        
        train_perplexity = math.exp(_loss) #log perplexity和交叉熵等价
        
        if _global_step % 10 == 0:
            logger.log('loss', _loss, _global_step)
            print(str(_global_step)+'\t', str(_loss)+'\t', str(train_perplexity)+'\t')
        
        if _global_step % 1000 == 0:
            save_path = saver.save(sess, save_dir + '/checkpoint', global_step=_global_step)
            print("Model saved in path: %s" % save_path)

iteration	 loss	 train_perplexity	
30	 4.9704647	 144.09383314412344	
40	 4.8477287	 127.45058607073132	
50	 5.0071187	 149.47343757107976	
60	 4.419165	 83.02694008171727	
70	 5.4537597	 233.6349069571184	
80	 5.0722723	 159.53643052981928	
90	 4.6207333	 101.56848104795321	
100	 4.8736887	 130.80251907423016	
110	 5.0162563	 150.84552989686424	
120	 4.570864	 96.62757931168598	
130	 4.6913524	 109.00048888484211	
140	 4.373732	 79.33918081160071	
150	 4.755118	 116.17734887870087	
160	 4.4949574	 89.5643576766001	
170	 4.5904145	 98.53526693816247	
180	 4.9017754	 134.52840416213456	


KeyboardInterrupt: 

### Compute perplexity on test set

In [8]:
inputs = []
targets = []

l = len(data_test_files)
for i in range(l): 
    while(True):
        x_test, y_test = get_data(hparams.n_time, data_test_files,'test',i, 'music')
        if(x_test.shape == y_test.shape):
            break       
    inputs.append(x_test)
    targets.append(y_test)
inputs = np.stack(inputs)
targets = np.stack(targets)

test_loss = sess.run(loss, feed_dict={X: inputs, Y: targets})
test_perplexity = math.exp(test_loss)
test_perplexity

99.23060071287466